In [ ]:
from pathlib import Path
import os


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "interface_analyzer" / "reproducibility").exists():
            return path
    raise RuntimeError("Could not find repository root containing interface_analyzer/reproducibility")


PROJECT_ROOT = find_repo_root()
REPRO_DIR = PROJECT_ROOT / "interface_analyzer" / "reproducibility"
DATASET_DIR = REPRO_DIR / "dataset"
LOCAL_OUTPUT_DIR = REPRO_DIR / "_local_outputs"
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Full manuscript-scale post-processing files are intentionally not bundled.
# Set INTERFACE_ANALYZER_DATA to the directory containing those generated files.
FULL_DATA_ROOT = Path(os.environ.get("INTERFACE_ANALYZER_DATA", LOCAL_OUTPUT_DIR)).expanduser()
FULL_DATA_ROOT.mkdir(parents=True, exist_ok=True)

# For quick local CFG tests this defaults to the bundled sample dataset.
CFG_DIR = Path(os.environ.get("INTERFACE_ANALYZER_CFG_DIR", DATASET_DIR)).expanduser()

print("Project root:", PROJECT_ROOT)
print("Bundled CFG dataset:", DATASET_DIR)
print("Analysis data root:", FULL_DATA_ROOT)
print("CFG input dir:", CFG_DIR)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

In [ ]:
# =========================
# Basic input parameters
# =========================

#925.1	582.9   86.3
#925.4	583.3   85.3
#925 	583.1   83.7

Lx = 582.9            # Interface length along x, unit: Å
dt_ps = 0.5             # Time interval between saved frames, unit: ps
dt_ns = dt_ps / 1000.0  # Convert to ns for kinetic fitting and plotting

Tm = 925.3              # Melting temperature, unit: K

# Volumetric latent heat: J/m^3 -> eV/Å^3
latent_heat_vol = 9.518e8 / 1.60218e11

# Interfacial stiffness: mJ/m^2 -> eV/Å^2
stiffness = 86.4 / 1.60218e4

# Gibbs-Thomson coefficient: Gamma = Tm * stiffness / latent_heat_vol
# Unit: Å·K
gamma_gt = Tm * stiffness / latent_heat_vol

print(f"latent_heat_vol = {latent_heat_vol:.6e} eV/Å^3")
print(f"stiffness       = {stiffness:.6e} eV/Å^2")
print(f"gamma_gt        = {gamma_gt:.6f} Å·K")

In [ ]:
from pathlib import Path
import pickle
# --- 1. Path setup ---
SAVE_DIR = FULL_DATA_ROOT / "100_010"
# Use a consistent variable name, Path_pkl
Path_pkl = SAVE_DIR / "100_010_cfg_post_orientation_grid_2_5_d_6_0.pkl"

with open(Path_pkl, "rb") as f:
        # Load the data back into the 'results_all' variable
        results_all = pickle.load(f)

timesteps = sorted(results_all.keys())

h_upper_all = np.array([results_all[ts]["h_upper"] for ts in timesteps], dtype=float)
h_lower_all = np.array([results_all[ts]["h_lower"] for ts in timesteps], dtype=float)

N_frames, N_bins = h_upper_all.shape

print(f"N_frames = {N_frames}")
print(f"N_bins   = {N_bins}")
print(f"h_upper shape = {h_upper_all.shape}")
print(f"h_lower shape = {h_lower_all.shape}")

In [ ]:
# =========================
# Fourier transform of interface fluctuations
# =========================

# Remove the instantaneous mean interface position from each frame
h_upper_centered = h_upper_all - np.mean(h_upper_all, axis=1, keepdims=True)
h_lower_centered = h_lower_all - np.mean(h_lower_all, axis=1, keepdims=True)

# Real FFT along the interface direction
A_upper = np.fft.rfft(h_upper_centered, axis=1, norm="forward")
A_lower = np.fft.rfft(h_lower_centered, axis=1, norm="forward")

# Wavevector values: k = 2*pi*n/Lx
dx = Lx / N_bins
k_vals = 2.0 * np.pi * np.fft.rfftfreq(N_bins, d=dx)  # unit: Å^-1

print(f"Number of k modes = {len(k_vals)}")
print(f"k_min (nonzero)   = {k_vals[1]:.6f} Å^-1")

In [ ]:
# =========================
# Fitting setup
# =========================

# k indices used for kinetic fitting
# Example: modes 7 to 13 inclusive
k_indices = np.arange(5, 18)

# Maximum lag used in the time-correlation analysis
max_lag = N_frames // 5
time_lags_ns = np.arange(max_lag) * dt_ns

# Relaxation-function fitting window
fit_min = 0.4
fit_max = 0.85
min_points = 5

print("Selected k modes:")
for i in k_indices:
    print(f"  i = {i:2d}, k = {k_vals[i]:.6f} Å^-1")

In [ ]:
# =========================
# Single-exponential relaxation model
# =========================

def relaxation_model(t, tau):
    return 1.0 - np.exp(-t / tau)

In [ ]:
# =========================
# Fit tau_k for each selected mode
# =========================

tau_results = []
k_results = []
fit_masks_all = {}
R_all = {}

plt.figure(figsize=(7, 6))

for j, i in enumerate(k_indices):
    # Combined statistics from upper and lower interfaces:
    # R(t) = sum <|A(k,t)-A(k,0)|^2> / sum 2<|A(k)|^2>
    numerator = np.zeros(max_lag, dtype=float)
    denominator = 0.0

    for A in [A_upper, A_lower]:
        A_k = A[:, i]
        mean_power = np.mean(np.abs(A_k)**2)
        denominator += 2.0 * mean_power

        for lag in range(max_lag):
            if lag == 0:
                diff = A_k - A_k
            else:
                diff = A_k[lag:] - A_k[:-lag]
            numerator[lag] += np.mean(np.abs(diff)**2)

    if denominator <= 0:
        print(f"Mode i={i}: zero denominator, skipped.")
        continue

    # Normalized mode relaxation function
    R = numerator / denominator
    R_all[i] = R

    # Once the curve reaches the upper cutoff, later points are excluded
    reached_upper = np.maximum.accumulate(R >= fit_max)
    fit_mask = (R > fit_min) & (~reached_upper) & np.isfinite(R)
    fit_masks_all[i] = fit_mask

    if np.sum(fit_mask) < min_points:
        print(f"Mode i={i}, k={k_vals[i]:.6f}: not enough fitting points.")
        continue

    try:
        # Initial guess for tau: choose a representative time scale in the fitting window
        t_fit = time_lags_ns[fit_mask]
        p0 = [max(t_fit[len(t_fit)//2], 1e-6)]

        popt, pcov = curve_fit(
            relaxation_model,
            time_lags_ns[fit_mask],
            R[fit_mask],
            p0=p0,
            bounds=(0.0, np.inf),
            maxfev=10000
        )

        tau_ns = popt[0]

        tau_results.append(tau_ns)
        k_results.append(k_vals[i])

        color = plt.cm.plasma(j / max(1, len(k_indices)-1))

        # Points excluded from fit
        plt.scatter(
            time_lags_ns[~fit_mask], R[~fit_mask],
            s=12, facecolors='none', edgecolors=color, alpha=0.35
        )

        # Points included in fit
        plt.scatter(
            time_lags_ns[fit_mask], R[fit_mask],
            s=18, color=color, alpha=0.85
        )

        # Fitted curve
        plt.plot(
            time_lags_ns,
            relaxation_model(time_lags_ns, tau_ns),
            color=color, lw=1.5,
            label=fr"$k={k_vals[i]:.4f}\ \AA^{{-1}},\ \tau={tau_ns:.4f}$ ns"
        )

    except Exception as e:
        print(f"Mode i={i}, k={k_vals[i]:.6f}: fitting failed. Error: {e}")

plt.axhline(fit_min, color='gray', linestyle=':', alpha=0.6)
plt.axhline(fit_max, color='gray', linestyle=':', alpha=0.6)

plt.xlabel("Time lag (ns)")
plt.ylabel("Normalized mode relaxation function")
plt.title("100_010: mode relaxation fitting")
plt.xlim(-0.002, 0.15)
plt.grid(True, alpha=0.25)
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib as mpl

mpl.rcParams["font.family"] = "serif"
mpl.rcParams["mathtext.fontset"] = "stix"

# =========================
# Fit tau_k for each selected mode
# =========================

tau_results = []
k_results = []
fit_masks_all = {}
R_all = {}

fig, ax = plt.subplots(figsize=(7.5, 7.5))

for j, i in enumerate(k_indices):
    # Combined statistics from upper and lower interfaces:
    # R(t) = sum <|A(k,t)-A(k,0)|^2> / sum 2<|A(k)|^2>
    numerator = np.zeros(max_lag, dtype=float)
    denominator = 0.0

    for A in [A_upper, A_lower]:
        A_k = A[:, i]
        mean_power = np.mean(np.abs(A_k)**2)
        denominator += 2.0 * mean_power

        for lag in range(max_lag):
            if lag == 0:
                diff = A_k - A_k
            else:
                diff = A_k[lag:] - A_k[:-lag]
            numerator[lag] += np.mean(np.abs(diff)**2)

    if denominator <= 0:
        print(f"Mode i={i}: zero denominator, skipped.")
        continue

    R = numerator / denominator
    R_all[i] = R

    reached_upper = np.maximum.accumulate(R >= fit_max)
    fit_mask = (R > fit_min) & (~reached_upper) & np.isfinite(R)
    fit_masks_all[i] = fit_mask

    if np.sum(fit_mask) < min_points:
        print(f"Mode i={i}, k={k_vals[i]:.6f}: not enough fitting points.")
        continue

    try:
        t_fit = time_lags_ns[fit_mask]
        p0 = [max(t_fit[len(t_fit)//2], 1e-6)]

        popt, pcov = curve_fit(
            relaxation_model,
            time_lags_ns[fit_mask],
            R[fit_mask],
            p0=p0,
            bounds=(0.0, np.inf),
            maxfev=10000
        )

        tau_ps = popt[0]*1000

        tau_results.append(tau_ps)
        k_results.append(k_vals[i])

        color = plt.cm.plasma(j / max(1, len(k_indices)-1))

        # Points excluded from fit
        ax.scatter(
            time_lags_ns[~fit_mask], R[~fit_mask],
            s=26, facecolors='none', edgecolors=color,
            linewidths=1.0, alpha=0.35
        )

        # Points included in fit
        ax.scatter(
            time_lags_ns[fit_mask], R[fit_mask],
            s=32, color=color, alpha=0.88
        )

        # Fitted curve
        ax.plot(
            time_lags_ns,
            relaxation_model(time_lags_ns, tau_ps*0.001),
            color=color, lw=2.0,
            label=fr"$k={k_vals[i]:.4f}\ \AA^{{-1}},\ \tau={tau_ps:.1f}$ ps"
        )

    except Exception as e:
        print(f"Mode i={i}, k={k_vals[i]:.6f}: fitting failed. Error: {e}")

ax.axhline(fit_min, color='gray', linestyle=':', lw=1.6, alpha=0.7)
ax.axhline(fit_max, color='gray', linestyle=':', lw=1.6, alpha=0.7)

ax.set_xlabel("Time lag (ns)", fontsize=20)
ax.set_ylabel("Normalized mode relaxation function", fontsize=20)

ax.set_xlim(-0.002, 0.15)
ax.tick_params(axis="both", labelsize=16, width=1.4, length=6)

for spine in ax.spines.values():
    spine.set_linewidth(1.4)

ax.grid(True, ls="--", alpha=0.30)

ax.legend(
    fontsize=12,
    frameon=False,
    loc="best",
    handlelength=2.2
)

plt.tight_layout()
plt.savefig("mode_relaxation_fitting.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
# =========================
# Summary of fitted tau_k
# =========================

k_array = np.array(k_results)
tau_array = np.array(tau_results)

print("Fitted relaxation times:")
for kk, tt in zip(k_array, tau_array):
    print(f"k = {kk:.6f} Å^-1, tau = {tt:.6f} ns")

In [ ]:
# =========================
# Fit kinetic coefficient mu
# =========================

if len(k_array) < 2:
    raise RuntimeError("Not enough valid tau(k) points for fitting mu.")

x = gamma_gt * k_array**2
y = 1.0 / tau_array   # unit: ns^-1

# Linear fit: y = mu_MD * x + intercept
mu_MD, intercept = np.polyfit(x, y, 1)

# Convert from Å/(ns·K) to m/(s·K)
mu_SI = mu_MD * 0.1

print(f"mu_MD      = {mu_MD:.6e} Å/(ns·K)")
print(f"intercept  = {intercept:.6e} ns^-1")
print(f"mu_SI      = {mu_SI:.6f} m/(s·K)")

In [ ]:
# =========================
# Plot tau_k vs k
# =========================

plt.figure(figsize=(7, 6))

plt.loglog(k_array, tau_array, 'ks', label='MD data')

k_ref = np.linspace(np.min(k_array)*0.9, np.max(k_array)*1.1, 200)
tau_ref = 1.0 / (mu_MD * gamma_gt * k_ref**2)

plt.loglog(
    k_ref, tau_ref, 'r-', lw=1.8,
    label=fr'Fit: $\mu={mu_SI:.3f}\ \mathrm{{m/(s\cdot K)}}$'
)

plt.xlabel(r"$k\ (\AA^{-1})$")
plt.ylabel(r"$\tau\ (\mathrm{ns})$")
plt.title("100_010: kinetic coefficient fitting")
plt.grid(True, which="both", alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Plot inverse relaxation rate vs Gamma * k^2
# =========================

plt.figure(figsize=(7, 6))

plt.plot(x, y, 'ko', label='MD data')

x_ref = np.linspace(0, np.max(x)*1.05, 200)
y_ref = mu_MD * x_ref + intercept

plt.plot(
    x_ref, y_ref, 'r-',
    label=fr'Linear fit: slope = {mu_MD:.4e} Å/(ns·K)'
)

plt.xlabel(r"$\Gamma k^2\ (\mathrm{\AA^{-1} K})$")
plt.ylabel(r"$1/\tau\ (\mathrm{ns^{-1}})$")
plt.title("100_010: linear fit for kinetic coefficient")
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# Optional: fit through the origin
# =========================

mu_MD_origin = np.sum(x * y) / np.sum(x * x)
mu_SI_origin = mu_MD_origin * 0.1

print(f"mu_MD (through origin) = {mu_MD_origin:.6e} Å/(ns·K)")
print(f"mu_SI (through origin) = {mu_SI_origin:.6f} m/(s·K)")
plt.figure(figsize=(7, 6))

plt.plot(x, y, 'ko', label='MD data')

x_ref = np.linspace(0, np.max(x)*1.05, 200)
y_free = mu_MD * x_ref + intercept
y_origin = mu_MD_origin * x_ref

plt.plot(x_ref, y_free, 'r-', label=fr'Free intercept: $\mu={mu_SI:.3f}$ m/(s·K)')
plt.plot(x_ref, y_origin, 'b--', label=fr'Through origin: $\mu={mu_SI_origin:.3f}$ m/(s·K)')

plt.xlabel(r"$\Gamma k^2\ (\mathrm{\AA^{-1} K})$")
plt.ylabel(r"$1/\tau\ (\mathrm{ns^{-1}})$")
plt.title("100_010: comparison of linear fits")
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
k_array = np.array(k_results)
tau_array = np.array(tau_results)
import numpy as np
import matplotlib.pyplot as plt

# x = Gamma * k^2
# y = 1 / tau
x_all = gamma_gt * k_array**2
y_all = 1.0 / tau_array   # ns^-1

# Keep the original k values for later display
k_all = k_array.copy()

# Sort by increasing k; this is usually already true, but keeps the workflow robust
sort_idx = np.argsort(k_all)
k_all = k_all[sort_idx]
x_all = x_all[sort_idx]
y_all = y_all[sort_idx]

print("Available fitted modes:")
for i, (kk, xx, yy) in enumerate(zip(k_all, x_all, y_all)):
    print(f"{i:2d}: k = {kk:.6f} Å^-1, Gamma*k^2 = {xx:.6e}, 1/tau = {yy:.6e}")

# =========================
# Scan all contiguous k-windows
# =========================

min_window_size = 4   # Minimum number of points required for fitting; adjust to 3 or 5 if needed
results_scan = []

n_points = len(k_all)

for start in range(n_points):
    for end in range(start + min_window_size, n_points + 1):
        # Python slicing: [start:end]
        k_win = k_all[start:end]
        x_win = x_all[start:end]
        y_win = y_all[start:end]

        # ---- Free-intercept fit: y = a*x + b ----
        slope_free, intercept_free = np.polyfit(x_win, y_win, 1)

        # ---- Through-origin fit: y = a*x ----
        slope_origin = np.sum(x_win * y_win) / np.sum(x_win * x_win)

        # ---- Convert slope to SI ----
        # slope unit: Å/(ns·K)
        mu_free_SI = slope_free * 0.1
        mu_origin_SI = slope_origin * 0.1

        # ---- Goodness-of-fit (free intercept) ----
        y_pred_free = slope_free * x_win + intercept_free
        ss_res_free = np.sum((y_win - y_pred_free)**2)
        ss_tot = np.sum((y_win - np.mean(y_win))**2)
        r2_free = 1 - ss_res_free / ss_tot if ss_tot > 0 else np.nan

        # ---- Goodness-of-fit (through origin) ----
        y_pred_origin = slope_origin * x_win
        ss_res_origin = np.sum((y_win - y_pred_origin)**2)
        r2_origin_like = 1 - ss_res_origin / ss_tot if ss_tot > 0 else np.nan
        # Note: this R2 is not the strict standard definition for a through-origin fit, but it is useful for relative comparison

        results_scan.append({
            "start": start,
            "end": end,
            "n_points": len(k_win),
            "k_min": k_win[0],
            "k_max": k_win[-1],
            "slope_free_MD": slope_free,
            "intercept_free": intercept_free,
            "mu_free_SI": mu_free_SI,
            "slope_origin_MD": slope_origin,
            "mu_origin_SI": mu_origin_SI,
            "r2_free": r2_free,
            "r2_origin_like": r2_origin_like,
        })

print(f"Scanned {len(results_scan)} windows.")

In [ ]:
# =========================
# Print scan results
# =========================

print(
    f"{'start':>5} {'end':>5} {'n':>3} "
    f"{'k_min':>10} {'k_max':>10} "
    f"{'mu_free':>10} {'mu_origin':>10} "
    f"{'intercept':>12} {'R2_free':>10}"
)

for row in results_scan:
    print(
        f"{row['start']:5d} {row['end']:5d} {row['n_points']:3d} "
        f"{row['k_min']:10.4f} {row['k_max']:10.4f} "
        f"{row['mu_free_SI']:10.4f} {row['mu_origin_SI']:10.4f} "
        f"{row['intercept_free']:12.4e} {row['r2_free']:10.4f}"
    )

In [ ]:
# =========================
# Build matrices for heatmaps
# =========================

mu_free_mat = np.full((n_points, n_points), np.nan)
mu_origin_mat = np.full((n_points, n_points), np.nan)
intercept_mat = np.full((n_points, n_points), np.nan)
r2_mat = np.full((n_points, n_points), np.nan)

for row in results_scan:
    i = row["start"]
    j = row["end"] - 1   # Use the last point index as the window endpoint
    mu_free_mat[i, j] = row["mu_free_SI"]
    mu_origin_mat[i, j] = row["mu_origin_SI"]
    intercept_mat[i, j] = row["intercept_free"]
    r2_mat[i, j] = row["r2_free"]

In [ ]:
plt.figure(figsize=(7, 6))
im = plt.imshow(mu_free_mat, origin='lower', aspect='auto')
plt.colorbar(im, label=r'$\mu$ (m/(s·K))')

plt.xlabel('End index')
plt.ylabel('Start index')
plt.title('Free-intercept fitted μ for each k-window')

plt.xticks(range(n_points), [f"{k:.3f}" for k in k_all], rotation=90)
plt.yticks(range(n_points), [f"{k:.3f}" for k in k_all])

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
im = plt.imshow(mu_origin_mat, origin='lower', aspect='auto')
plt.colorbar(im, label=r'$\mu$ (m/(s·K))')

plt.xlabel('End index')
plt.ylabel('Start index')
plt.title('Through-origin fitted μ for each k-window')

plt.xticks(range(n_points), [f"{k:.3f}" for k in k_all], rotation=90)
plt.yticks(range(n_points), [f"{k:.3f}" for k in k_all])

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))
im = plt.imshow(intercept_mat, origin='lower', aspect='auto')
plt.colorbar(im, label=r'Intercept (ns$^{-1}$)')

plt.xlabel('End index')
plt.ylabel('Start index')
plt.title('Intercept from free-intercept fit')

plt.xticks(range(n_points), [f"{k:.3f}" for k in k_all], rotation=90)
plt.yticks(range(n_points), [f"{k:.3f}" for k in k_all])

plt.tight_layout()
plt.show()